In [1]:
import argparse
import itertools
import os
import pathlib
import sys
from functools import reduce

import duckdb
import pandas as pd
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm
profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir  # default to root_dir instead of NAS

In [2]:
patient_id_file = pathlib.Path(f"{profile_base_dir}/data/patient_IDs.txt").resolve(
    strict=True
)
patients = pd.read_csv(
    patient_id_file, header=None, names=["patient_id"]
).patient_id.tolist()

In [3]:
out_dict = {
    "file_path": [],
    "patient_id": [],
    "well_fov": [],
    "feature_type": [],
    "compartment": [],
    # "df_shape": [],
}

# get all well_fovs for a patient
for patient in tqdm.tqdm(patients, desc="Processing patients", leave=True):
    patient_dir = profile_base_dir / "data" / patient / "extracted_features"
    well_fovs = patient_dir.glob("*")  # get all well_fovs for a patient
    # print(f"Found well_fovs: {well_fovs}")
    for well_fov in tqdm.tqdm(well_fovs, desc="Processing well_fovs", leave=False):
        if "stats" in well_fov.stem:
            continue
        features = pathlib.Path(well_fov).glob("*.parquet")
        for feature in features:
            feature_type = feature.stem.split("_")[2]
            compartment = feature.stem.split("_")[0]
            out_dict["file_path"].append(feature)
            out_dict["patient_id"].append(patient)
            out_dict["well_fov"].append(feature.parent.stem)
            out_dict["feature_type"].append(feature_type)
            out_dict["compartment"].append(compartment)
            # out_dict["df_shape"].append(pd.read_parquet(feature).shape)
df = pd.DataFrame(out_dict)
# df = df.loc[df["patient_id"] == "NF0014_T1"]
df = df.loc[(df["feature_type"] == "Granularity") & (df["compartment"] != "Organoid")]
df = df.loc[(df["compartment"] != "Organoid")]

df.head()

Processing patients:   0%|          | 0/13 [00:00<?, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

...

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

,file_path,patient_id,well_fov,feature_type,compartment
1,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell
5,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cytoplasm
16,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cytoplasm
23,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell
27,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell


In [4]:
from tqdm import tqdm

tqdm.pandas()


def safe_read_shape(x):
    try:
        df = pd.read_parquet(x)
        return df.shape, df.isna().sum().sum()
    except Exception as e:
        print(f"Error reading {x}: {e}")
        return None, None


# if not pathlib.Path("../logs/feature_file_info.parquet").exists():
df[["df_shape", "missing_values"]] = df["file_path"].progress_apply(
    lambda x: pd.Series(safe_read_shape(x))
)
df["file_path"] = df["file_path"].astype(str)
df.to_parquet("../logs/feature_file_info.parquet", index=False)
# else:
#     df = pd.read_parquet("../logs/feature_file_info.parquet")

100%|██████████| 39627/39627 [01:13<00:00, 542.52it/s]


In [5]:
df.sort_values(["patient_id", "well_fov"], inplace=True)
df.reset_index(drop=True, inplace=True)
df

,file_path,patient_id,well_fov,feature_type,compartment,df_shape,missing_values
0,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cell,"(13, 18)",0
1,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cytoplasm,"(13, 18)",0
2,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cytoplasm,"(13, 18)",0
3,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cell,"(13, 18)",0
4,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cell,"(13, 18)",0
...,...,...,...,...,...,...,...
39622,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Nuclei,"(18, 18)",0
39623,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Cytoplasm,"(18, 18)",0
39624,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Nuclei,"(18, 18)",0
39625,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Nuclei,"(18, 18)",0


In [6]:
# merge the cells, cytoplasm, and whole cell features for a given well_fov and patient_id
# check for missing values and shape of the dataframes
out_dict = {
    "patient_id": [],
    "well_fov": [],
    "path": [],
    "type": [],
    "feature_type": [],
}
for row in tqdm(
    df.itertuples(), total=df.shape[0], desc="Merging features", leave=True
):
    out_dict["patient_id"].append(row.patient_id)
    out_dict["well_fov"].append(row.well_fov)
    out_dict["path"].append(row.file_path)
    out_dict["type"].append(f"{row.compartment}")
    out_dict["feature_type"].append(row.feature_type)
out_df = pd.DataFrame(out_dict)
out_df.drop_duplicates(subset=["patient_id", "well_fov", "type"], inplace=True)
# pivot such that each type has its own column
out_df = out_df.pivot(
    index=[
        "patient_id",
        "well_fov",
    ],
    columns="type",
    values="path",
).reset_index()

Merging features: 100%|██████████| 39627/39627 [00:00<00:00, 728380.48it/s]


In [7]:
out_df

type,patient_id,well_fov,Cell,Cytoplasm,Nuclei
0,NF0014_T1,C10-1,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
1,NF0014_T1,C10-2,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
2,NF0014_T1,C11-1,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
3,NF0014_T1,C11-2,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
4,NF0014_T1,C2-1,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
...,...,...,...,...,...
3333,SARCO361_T1,G9-3,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
3334,SARCO361_T1,G9-4,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
3335,SARCO361_T1,G9-5,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
3336,SARCO361_T1,G9-6,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...


In [8]:
labels_dict = {
    "Cell_labels": [],
    "Cytoplasm_labels": [],
    "Nuclei_labels": [],
    "patient_id": [],
    "well_fov": [],
}

# merge the dataframes and check for missing values and shape
for row in tqdm(
    out_df.itertuples(),
    total=out_df.shape[0],
    desc="Checking merged features",
    leave=True,
):
    try:
        cell_df = pd.read_parquet(row.Cell)
        cytoplasm_df = pd.read_parquet(row.Cytoplasm)
        nuclei_df = pd.read_parquet(row.Nuclei)
        labels_dict["Cell_labels"].append(cell_df["object_id"].tolist())
        labels_dict["Cytoplasm_labels"].append(cytoplasm_df["object_id"].tolist())
        labels_dict["Nuclei_labels"].append(nuclei_df["object_id"].tolist())
        labels_dict["patient_id"].append(row.patient_id)
        labels_dict["well_fov"].append(row.well_fov)
    except Exception as e:
        print(f"Error reading files for {row.patient_id} {row.well_fov}: {e}")
labels_df = pd.DataFrame(labels_dict)

Checking merged features:  81%|████████  | 2702/3338 [00:11<00:02, 249.39it/s]

Error reading files for NF0040_T1 B10-4: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B11-7: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B2-3: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B3-1: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B3-4: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B3-5: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B3-6: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B4-5: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B5-2: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B5-5: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B7-2: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B7-5: cannot construct a FileSource from nan
Error reading files for NF0040_T1 B8-2: cannot con

Checking merged features:  83%|████████▎ | 2763/3338 [00:12<00:02, 275.08it/s]

Error reading files for NF0040_T1 D7-2: cannot construct a FileSource from nan
Error reading files for NF0040_T1 D7-3: cannot construct a FileSource from nan
Error reading files for NF0040_T1 D8-1: cannot construct a FileSource from nan
Error reading files for NF0040_T1 D8-2: cannot construct a FileSource from nan
Error reading files for NF0040_T1 D9-3: cannot construct a FileSource from nan
Error reading files for NF0040_T1 D9-5: cannot construct a FileSource from nan
Error reading files for NF0040_T1 E10-3: cannot construct a FileSource from nan
Error reading files for NF0040_T1 E10-5: cannot construct a FileSource from nan
Error reading files for NF0040_T1 E10-7: cannot construct a FileSource from nan
Error reading files for NF0040_T1 E11-5: cannot construct a FileSource from nan
Error reading files for NF0040_T1 E3-1: cannot construct a FileSource from nan
Error reading files for NF0040_T1 E3-7: cannot construct a FileSource from nan
Error reading files for NF0040_T1 E4-4: cannot c

Checking merged features: 100%|██████████| 3338/3338 [00:14<00:00, 227.61it/s]


In [9]:
labels_df["labels_match"] = labels_df.apply(
    lambda row: row["Cell_labels"] == row["Nuclei_labels"],
    axis=1,
)
labels_df["same_number_of_labels"] = labels_df.apply(
    lambda row: len(row["Cell_labels"]) == len(row["Nuclei_labels"]),
    axis=1,
)
labels_df["unique_labels_across_compartments"] = labels_df.apply(
    lambda row: set(row["Nuclei_labels"]) - set(row["Cytoplasm_labels"]), axis=1
)
labels_df["nuc_cyto_unique"] = labels_df.apply(
    lambda row: set(row["Nuclei_labels"]) - set(row["Cytoplasm_labels"]), axis=1
)
labels_df["nuc_cell_unique"] = labels_df.apply(
    lambda row: set(row["Nuclei_labels"]) - set(row["Cell_labels"]), axis=1
)
labels_df["cyto_cell_unique"] = labels_df.apply(
    lambda row: set(row["Cytoplasm_labels"]) - set(row["Cell_labels"]), axis=1
)
labels_df.loc[labels_df["labels_match"] == False]

,Cell_labels,Cytoplasm_labels,Nuclei_labels,patient_id,well_fov,labels_match,same_number_of_labels,unique_labels_across_compartments,nuc_cyto_unique,nuc_cell_unique,cyto_cell_unique
14,"[1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 1...","[1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 1...","[1, 2, 3, 4, 6, 8, 9, 10, 11, 12, 13, 14, 15, ...",NF0014_T1,C7-1,False,False,{},{},{},{}
21,"[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0014_T1,D10-2,False,False,"{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{}
22,"[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0014_T1,D11-1,False,False,"{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{}
32,"[3, 4, 5, 6, 7, 8, 9, 10]","[3, 4, 5, 6, 7, 8, 9, 10]","[3, 4, 6, 7, 8, 9, 10]",NF0014_T1,D5-1,False,False,{},{},{},{}
40,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0014_T1,D9-1,False,False,{},{},{},{}
...,...,...,...,...,...,...,...,...,...,...,...
2672,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0040_T1,B2-7,False,False,{},{},{},{}
2681,"[1, 2, 3, 5, 6, 7, 10, 11, 12, 14, 15, 16, 18,...","[1, 2, 3, 5, 6, 7, 10, 11, 12, 14, 15, 16, 18,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0040_T1,C8-2,False,False,"{66, 4, 8, 9, 41, 13, 17}","{66, 4, 8, 9, 41, 13, 17}","{66, 4, 8, 9, 41, 13, 17}",{}
2683,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0040_T1,D5-7,False,False,"{58, 69, 54, 63}","{58, 69, 54, 63}","{58, 69, 54, 63}",{}
2684,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]","[1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12]",NF0040_T1,D6-1,False,False,{},{},{},{}


In [10]:
# get the patient_id and well_fov for the rows where the labels do not match and check the corresponding feature files for those rows
mismatched_labels = labels_df.loc[
    labels_df["labels_match"] == False, ["patient_id", "well_fov"]
]
mismatched_labels
for row in tqdm(
    mismatched_labels.itertuples(),
    total=mismatched_labels.shape[0],
    desc="Checking mismatched labels",
    leave=True,
):
    patient_id = row.patient_id
    well_fov = row.well_fov
    print(f"cd ../../{patient_id}/segmentation_masks/ ; rm -r {well_fov}")
    print(f"cd ../../{patient_id}/extracted_features/ ; rm -r {well_fov}")

    continue

Checking mismatched labels: 100%|██████████| 187/187 [00:00<00:00, 82353.51it/s]

cd ../../NF0014_T1/segmentation_masks/ ; rm -r C7-1
cd ../../NF0014_T1/extracted_features/ ; rm -r C7-1
cd ../../NF0014_T1/segmentation_masks/ ; rm -r D10-2
cd ../../NF0014_T1/extracted_features/ ; rm -r D10-2
cd ../../NF0014_T1/segmentation_masks/ ; rm -r D11-1
cd ../../NF0014_T1/extracted_features/ ; rm -r D11-1
cd ../../NF0014_T1/segmentation_masks/ ; rm -r D5-1
cd ../../NF0014_T1/extracted_features/ ; rm -r D5-1
cd ../../NF0014_T1/segmentation_masks/ ; rm -r D9-1
cd ../../NF0014_T1/extracted_features/ ; rm -r D9-1
cd ../../NF0014_T1/segmentation_masks/ ; rm -r D9-2
cd ../../NF0014_T1/extracted_features/ ; rm -r D9-2
cd ../../NF0014_T1/segmentation_masks/ ; rm -r E8-2
cd ../../NF0014_T1/extracted_features/ ; rm -r E8-2
cd ../../NF0014_T1/segmentation_masks/ ; rm -r F10-1
cd ../../NF0014_T1/extracted_features/ ; rm -r F10-1
cd ../../NF0014_T1/segmentation_masks/ ; rm -r F4-1
cd ../../NF0014_T1/extracted_features/ ; rm -r F4-1
cd ../../NF0014_T1/segmentation_masks/ ; rm -r G4-2
cd ../

In [ ]:
labels_df.loc[labels_df["same_number_of_labels"] == False].value_counts("patient_id")

In [ ]:
# parse through all feature files and find any files that have nas for all of a column


In [ ]:
patient_ids = pd.read_csv(
    pathlib.Path(f"{root_dir}/data/patient_IDs.txt").resolve(strict=True),
    header=None,
    names=["patient_id"],
).patient_id.tolist()

In [ ]:
import tqdm.notebook as tqdm

all_nans = []
for patient in tqdm.tqdm(
    patient_ids, desc="Checking for missing values in features", leave=True
):
    patient_dir = profile_base_dir / "data" / patient / "extracted_features"
    well_fovs = patient_dir.glob("*")  # get all well_fovs for a patient
    for well_fov in tqdm.tqdm(
        well_fovs, desc=f"Checking {patient} for missing values", leave=False
    ):
        if "stats" in well_fov.stem:
            continue
        features = pathlib.Path(well_fov).glob("*.parquet")
        for feature in features:
            if "sam" not in feature.stem.lower():
                continue
            try:
                df = pd.read_parquet(feature)
                if df.isna().all().any():
                    all_nans.append(feature)
            except Exception as e:
                print(f"Error reading {feature}: {e}")

In [ ]:
all_nans

In [ ]:
all_nans[0]